# Advanced Python OOP: Method Overriding — 28 Problems with Complete Solutions

**Scope:** inheritance, method resolution, dynamic dispatch, `__str__` and `__repr__`, `super()`, cooperative multiple inheritance, descriptor and special-method lookup, overriding contracts, and capstone designs.

**Based on the supplied lecture:** its `Person`/`Student` representation examples and `Shape`/`Polygon` inherited-method examples are the conceptual starting point. All examples and exercises here are newly constructed and substantially extended. The original source illustrates that an inherited `Shape.extended_info()` calls the overridden `Polygon.info()` via `self`, and that an inherited `Person.__str__` can delegate to a subclass's `__repr__`.

**Audience:** comfortable with classes, functions, exceptions, and basic inheritance. **Runtime:** Python 3.10+; standard library only; no network, installation, files, or external data needed.

**How to study:** read each **Your task** section; predict the output or implement the design *before* inspecting its **Worked solution**; run the solution cell, then the separate assertion cell. Cells tagged `solution` can be hidden using a notebook interface that supports cell tags. Run **Kernel → Restart & Run All** to verify every independent exercise. A `PASS` line means all assertions in that problem succeeded; assertions are not formal proof.

**Best practices applied:** explicit contracts and edge cases, short focused methods, meaningful `super()` usage, stable deterministic tests, descriptive representations, correct use of `NotImplemented`, and cooperative initialization where multiple inheritance is intended. Worked examples intentionally include a few *bad patterns* only to demonstrate how to correct them. The source lesson's Python 3.6-era metadata does not constrain these new Python 3.10+ exercises.


## Reference card: lookup rules worth memorizing

1. `str(obj)` invokes the runtime type's `__str__` special method (not an arbitrary instance attribute). If no subclass overrides it, an inherited method wins. When `object.__str__` is in effect, it calls the object's `__repr__`.
2. `repr(obj)` invokes the runtime type's `__repr__`. Containers usually display element `repr` values; `print(obj)` uses `str(obj)`.
3. Ordinary `self.method()` performs fresh attribute lookup on the runtime object; a method inherited from a parent can therefore call a method overridden by a child.
4. `super().method()` begins looking **after the class where `super()` is used** in the runtime class's method resolution order (MRO). It does not mean “always call my immediate lexical parent.”
5. In a cooperative diamond, every participating method should accept/forward arguments consistently and call `super()` **once**; the final base terminates the chain.
6. When overriding, preserve documented parameter/return/exception behavior required by callers. Python does not automatically enforce the Liskov substitution principle.
7. Special methods such as `__str__` are looked up on the **type**, so assigning `instance.__str__` does not change `str(instance)`.

**Important subtlety:** `object.__str__` uses the representation fallback; an explicitly inherited `Person.__str__` does *not* automatically delegate to a child's `__repr__` unless its implementation does so. Do not confuse the two cases.


In [1]:
import sys

assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
print(f"Python {sys.version_info.major}.{sys.version_info.minor}: notebook ready")


Python 3.13: notebook ready


---

## Part I — Representations and dynamic dispatch

Start with the two source-lesson themes: method lookup and `__str__` / `__repr__`; move beyond them into actual call paths and protocols.


### Problem 01 — Build the representation-resolution matrix · Advanced

**Concepts:** `object.__str__`, `__repr__`, deterministic tests

**Your task**

Create four classes: `P01Default`, `P01Repr`, `P01Str`, and `P01Both`. Override neither, only `__repr__`, only `__str__`, and both respectively. Predict `str`/`repr` for every instance without relying on unstable memory addresses. Explain which fallback is real.

**Hint:** For a class overriding only `__str__`, ask whether `repr()` calls `str()` (it does not).

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 01**

`str(x)` uses the type's `__str__`. When that is `object.__str__`, it defers to the type's `__repr__`. The reverse fallback does **not** exist: a class with only `__str__` still inherits the default `object.__repr__`. Default object representations contain runtime-specific addresses, so test only stable substrings.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [2]:
class P01Default:
    pass


class P01Repr:
    def __repr__(self) -> str:
        return "P01Repr()"


class P01Str:
    def __str__(self) -> str:
        return "human-readable"


class P01Both:
    def __repr__(self) -> str:
        return "P01Both()"

    def __str__(self) -> str:
        return "friendly P01Both"


p01_cases = [P01Default(), P01Repr(), P01Str(), P01Both()]
for item in p01_cases:
    print(f"{type(item).__name__}: str={str(item)!r}; repr={repr(item)!r}")


P01Default: str='<__main__.P01Default object at 0x000001EA8B444440>'; repr='<__main__.P01Default object at 0x000001EA8B444440>'
P01Repr: str='P01Repr()'; repr='P01Repr()'
P01Str: str='human-readable'; repr='<__main__.P01Str object at 0x000001EA9A43E900>'
P01Both: str='friendly P01Both'; repr='P01Both()'


In [3]:
assert "P01Default object at" in str(p01_cases[0])
assert repr(p01_cases[1]) == str(p01_cases[1]) == "P01Repr()"
assert str(p01_cases[2]) == "human-readable"
assert "P01Str object at" in repr(p01_cases[2])
assert str(p01_cases[3]) == "friendly P01Both"
assert repr(p01_cases[3]) == "P01Both()"
print("PASS Problem 01: Build the representation-resolution matrix")


PASS Problem 01: Build the representation-resolution matrix


### Problem 02 — An inherited `__str__` defeats a child `__repr__` · Advanced

**Concepts:** explicit inherited overrides, `repr` for containers

**Your task**

A base defines `__str__`; its child defines only `__repr__`. Predict both display functions. Then create a second base whose `__str__` *explicitly* delegates to `repr(self)`, and demonstrate the changed output. What happens in a list containing the child?

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 02**

An inherited explicit `__str__` is selected before `object.__str__` would be considered. Only the delegating base's implementation triggers the child's `__repr__`. A list represents its elements using `repr`, rather than their `str`.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [4]:
class P02FixedBase:
    def __str__(self) -> str:
        return "base string"


class P02Child(P02FixedBase):
    def __repr__(self) -> str:
        return "P02Child()"


class P02DelegatingBase:
    def __str__(self) -> str:
        return repr(self)


class P02DelegatingChild(P02DelegatingBase):
    def __repr__(self) -> str:
        return "P02DelegatingChild()"


p02_fixed, p02_delegating = P02Child(), P02DelegatingChild()
print(str(p02_fixed), repr(p02_fixed))
print(str(p02_delegating), repr([p02_fixed]))


base string P02Child()
P02DelegatingChild() [P02Child()]


In [5]:
assert str(p02_fixed) == "base string"
assert repr(p02_fixed) == "P02Child()"
assert str(p02_delegating) == "P02DelegatingChild()"
assert repr([p02_fixed]) == "[P02Child()]"
print("PASS Problem 02: An inherited `__str__` defeats a child `__repr__`")


PASS Problem 02: An inherited `__str__` defeats a child `__repr__`


### Problem 03 — Trace the inherited template method · Advanced

**Concepts:** template method, runtime `self`, intentional incomplete hooks

**Your task**

Implement `P03Shape.describe` once in a parent. It must call `self.area()` and `self.label()` without knowing subclasses. Create a rectangle and a circle that each override both hooks. Test that the inherited method uses the child methods and keeps the prescribed string shape.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 03**

Although `describe`'s code lives in `P03Shape`, `self` remains the actual rectangle/circle instance. Both virtual calls use the runtime object's method lookup. The base's hook methods raise `NotImplementedError` so incomplete implementations fail visibly.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [6]:
from math import pi


class P03Shape:
    def area(self) -> float:
        raise NotImplementedError("subclasses supply area()")

    def label(self) -> str:
        raise NotImplementedError("subclasses supply label()")

    def describe(self) -> str:
        return f"{self.label()}: {self.area():.2f} square units"


class P03Rectangle(P03Shape):
    def __init__(self, width: float, height: float) -> None:
        self.width, self.height = width, height

    def area(self) -> float:
        return self.width * self.height

    def label(self) -> str:
        return "Rectangle"


class P03Circle(P03Shape):
    def __init__(self, radius: float) -> None:
        self.radius = radius

    def area(self) -> float:
        return pi * self.radius**2

    def label(self) -> str:
        return "Circle"


p03_rect, p03_circle = P03Rectangle(3, 4), P03Circle(2)
print(p03_rect.describe())
print(p03_circle.describe())


Rectangle: 12.00 square units
Circle: 12.57 square units


In [7]:
assert p03_rect.describe() == "Rectangle: 12.00 square units"
assert p03_circle.describe() == f"Circle: {4 * pi:.2f} square units"
try:
    P03Shape().describe()
except NotImplementedError as exc:
    assert "label" in str(exc)
else:
    raise AssertionError("The abstract base hook must fail")
print("PASS Problem 03: Trace the inherited template method")


PASS Problem 03: Trace the inherited template method


### Problem 04 — Avoid infinite recursion in a method override · Advanced

**Concepts:** recursive dispatch, extending base behavior, `super()`

**Your task**

An override of `describe` should prefix the base description with `[audited] `. Show the incorrect conceptual pattern `return self.describe()` (do not actually run unbounded recursion), then implement and test the safe version using `super()`. Verify base information is produced once.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 04**

Calling `self.describe()` from `describe` re-selects the same child override, causing recursion. `super().describe()` advances to the next method in the MRO, while the parent's internal `self.details()` remains dynamically dispatched to the child's implementation.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [8]:
class P04Document:
    def details(self) -> str:
        return "generic"

    def describe(self) -> str:
        return f"document: {self.details()}"


class P04Invoice(P04Document):
    def __init__(self, number: int) -> None:
        self.number = number
        self.detail_calls = 0

    def details(self) -> str:
        self.detail_calls += 1
        return f"invoice #{self.number}"

    def describe(self) -> str:
        # WRONG: return self.describe()   # calls this override again
        return "[audited] " + super().describe()


p04_invoice = P04Invoice(17)
p04_text = p04_invoice.describe()
print(p04_text)


[audited] document: invoice #17


In [9]:
assert p04_text == "[audited] document: invoice #17"
assert p04_invoice.detail_calls == 1
assert P04Document().describe() == "document: generic"
print("PASS Problem 04: Avoid infinite recursion in a method override")


PASS Problem 04: Avoid infinite recursion in a method override


### Problem 05 — Construct safe and useful `repr` values · Advanced

**Concepts:** representation contracts, `!r`, containers

**Your task**

Implement a `P05Point` with a debugging `__repr__` suitable for reconstructing a simple object and a concise `__str__` suitable for a UI. Use `!r` to quote values safely, and distinguish `print(point)` from `print([point])`. Test quotes in a label.

**Hint:** Focus on round-tripping this deliberately simple constructor; representation round-tripping is a design goal, but never use `eval` on untrusted representations.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 05**

A debugging representation should identify the class and preserve unambiguous literal-like values. `!r` escapes embedded quotes. A concise string can intentionally differ. A container invokes the element's `repr`, which prevents a UI-oriented `str` from hiding useful debugging state.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [10]:
class P05Point:
    def __init__(self, x: int, y: int, label: str) -> None:
        self.x, self.y, self.label = x, y, label

    def __repr__(self) -> str:
        return f"P05Point({self.x!r}, {self.y!r}, {self.label!r})"

    def __str__(self) -> str:
        return f"{self.label}: ({self.x}, {self.y})"


p05_point = P05Point(2, -4, "O'Reilly")
print(str(p05_point))
print(repr(p05_point))
print([p05_point])


O'Reilly: (2, -4)
P05Point(2, -4, "O'Reilly")
[P05Point(2, -4, "O'Reilly")]


In [11]:
assert str(p05_point) == "O'Reilly: (2, -4)"
assert repr(p05_point) == f"P05Point(2, -4, {p05_point.label!r})"
assert repr([p05_point]) == '[' + repr(p05_point) + ']'
assert repr(p05_point).startswith("P05Point(2, -4, ")
print("PASS Problem 05: Construct safe and useful `repr` values")


PASS Problem 05: Construct safe and useful `repr` values


---

## Part II — Correct single-inheritance overrides

Protect initialization invariants, validate new behavior, and recognize that not every parent method should be skipped.


### Problem 06 — Initialize the base exactly once · Advanced

**Concepts:** base invariants, `super().__init__`, validation

**Your task**

A `P06Account` validates a nonempty identifier and initializes `events`. Implement a child `P06Savings` that adds an interest rate without duplicating base validation. Test bad inputs and show that events work. What invariant breaks if the child forgets `super().__init__`?

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 06**

Call `super().__init__(identifier)` first to establish the parent contract, then validate child state. Duplicating parent initialization risks drifting validation and forgetting internal fields (`events`). Using `ValueError` gives an explicit documented error for invalid user data.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [12]:
class P06Account:
    def __init__(self, identifier: str) -> None:
        if not isinstance(identifier, str) or not identifier.strip():
            raise ValueError("identifier must be nonempty")
        self.identifier = identifier
        self.events: list[str] = []

    def record(self, event: str) -> None:
        self.events.append(event)


class P06Savings(P06Account):
    def __init__(self, identifier: str, annual_rate: float) -> None:
        super().__init__(identifier)
        if not 0 <= annual_rate <= 1:
            raise ValueError("annual_rate must be between 0 and 1")
        self.annual_rate = annual_rate

    def record(self, event: str) -> None:
        super().record(f"SAVINGS: {event}")


p06_account = P06Savings("A-001", 0.04)
p06_account.record("deposit")
print(p06_account.identifier, p06_account.events)


A-001 ['SAVINGS: deposit']


In [13]:
assert p06_account.events == ["SAVINGS: deposit"]
assert p06_account.annual_rate == 0.04
for make_bad in (lambda: P06Savings("", 0.04),
                 lambda: P06Savings("valid", 1.2)):
    try:
        make_bad()
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid account data should fail")
print("PASS Problem 06: Initialize the base exactly once")


PASS Problem 06: Initialize the base exactly once


### Problem 07 — Override a property without breaking validation · Advanced

**Concepts:** property override, descriptor preservation, invariants

**Your task**

A base temperature property owns Celsius storage and guards against temperatures below absolute zero. Override only the setter in a child to additionally enforce a laboratory ceiling. Keep the inherited getter and setter's original guard. Explain why direct `_celsius` assignment is incorrect.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 07**

A `property` is a descriptor. The overriding subclass property is created by decorating its setter with `@P07Temperature.celsius.setter`, which retains the inherited getter. Delegate writes to the base property's setter through `P07Temperature.celsius.fset(self, value)`; direct storage would bypass base validation. Using the explicit descriptor is appropriate here because `super().celsius = value` is not a viable property-setter idiom.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [14]:
class P07Temperature:
    def __init__(self, celsius: float) -> None:
        self.celsius = celsius

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float) -> None:
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = float(value)


class P07LabTemperature(P07Temperature):
    @P07Temperature.celsius.setter
    def celsius(self, value: float) -> None:
        if value > 150:
            raise ValueError("above laboratory ceiling")
        P07Temperature.celsius.fset(self, value)


p07_temp = P07LabTemperature(20)
p07_temp.celsius = 100
print(p07_temp.celsius)


100.0


In [15]:
assert p07_temp.celsius == 100.0
for invalid in (-300, 151):
    try:
        p07_temp.celsius = invalid
    except ValueError:
        pass
    else:
        raise AssertionError(f"Must reject {invalid}")
assert p07_temp.celsius == 100.0
assert P07Temperature(-273.15).celsius == -273.15
print("PASS Problem 07: Override a property without breaking validation")


PASS Problem 07: Override a property without breaking validation


### Problem 08 — Use polymorphic classmethod constructors · Advanced

**Concepts:** `cls` versus hardcoded parent, overriding factories

**Your task**

Write a base classmethod `from_csv` that receives `cls`, builds an instance with three fields, and calls the selected class's constructor. A subclass should add a discount while reusing inherited parsing. Ensure the factory returns the subclass, not an accidentally hardcoded base type.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 08**

An inherited classmethod receives the subclass as `cls` when invoked through the subclass. Returning `cls(...)` preserves polymorphism. Design constructors so both types accept parsed base fields, with the child providing a default for extra fields. Hardcoding `P08Product(...)` inside `from_csv` would slice away the subclass.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [16]:
class P08Product:
    def __init__(self, sku: str, title: str, price: float) -> None:
        if price < 0:
            raise ValueError("price must be nonnegative")
        self.sku, self.title, self.price = sku, title, float(price)

    @classmethod
    def from_csv(cls, line: str):
        sku, title, price_text = line.split(",", maxsplit=2)
        return cls(sku.strip(), title.strip(), float(price_text))

    def final_price(self) -> float:
        return self.price


class P08SaleProduct(P08Product):
    def __init__(self, sku: str, title: str, price: float,
                 discount: float = 0.10) -> None:
        super().__init__(sku, title, price)
        if not 0 <= discount <= 1:
            raise ValueError("discount must be between 0 and 1")
        self.discount = discount

    def final_price(self) -> float:
        return self.price * (1 - self.discount)


p08_item = P08SaleProduct.from_csv("SKU9, Pen, 10.0")
print(type(p08_item).__name__, p08_item.final_price())


P08SaleProduct 9.0


In [17]:
assert type(p08_item) is P08SaleProduct
assert p08_item.final_price() == 9.0
assert type(P08Product.from_csv("SKU1,Pencil,2.0")) is P08Product
assert p08_item.sku == "SKU9"
print("PASS Problem 08: Use polymorphic classmethod constructors")


PASS Problem 08: Use polymorphic classmethod constructors


### Problem 09 — Static-method overrides and runtime lookup · Advanced

**Concepts:** staticmethod overriding, inherited callers

**Your task**

A base `quote` calls `self.format_amount`. Override `format_amount` as a staticmethod in a child. Predict whether the base `quote` uses the new formatting. Include two objects to show the inherited method changes behavior only for the child.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 09**

Accessing `self.format_amount` resolves against the runtime class even though it is a staticmethod. Static methods do not receive `self` or `cls` automatically, but a method of that name can still be overridden in a subclass. An inherited `quote` can thus customize itself through the override.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [18]:
class P09Receipt:
    @staticmethod
    def format_amount(amount: float) -> str:
        return f"{amount:.2f}"

    def quote(self, amount: float) -> str:
        return "Total: " + self.format_amount(amount)


class P09EuroReceipt(P09Receipt):
    @staticmethod
    def format_amount(amount: float) -> str:
        return f"EUR {amount:.2f}"


p09_base, p09_euro = P09Receipt(), P09EuroReceipt()
print(p09_base.quote(12.5), "|", p09_euro.quote(12.5))


Total: 12.50 | Total: EUR 12.50


In [19]:
assert p09_base.quote(12.5) == "Total: 12.50"
assert p09_euro.quote(12.5) == "Total: EUR 12.50"
assert P09EuroReceipt.format_amount(1) == "EUR 1.00"
print("PASS Problem 09: Static-method overrides and runtime lookup")


PASS Problem 09: Static-method overrides and runtime lookup


### Problem 10 — A cooperative `@classmethod` using `super()` · Advanced

**Concepts:** classmethod, `cls`, zero-argument `super()`

**Your task**

The parent class's `describe_class` returns the runtime class name, and the child's override adds a suffix without losing the runtime name. Call the method on the base, child, and grandchild. Explain how `cls` and `super()` interact.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 10**

For classmethods, an inherited method still receives the class used for access as `cls`. Zero-argument `super()` in the overriding classmethod begins after the defining class in the MRO, while the inherited parent method sees `cls` set to the actual subclass/grandchild.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [20]:
class P10Animal:
    @classmethod
    def describe_class(cls) -> str:
        return f"species={cls.__name__}"


class P10Bird(P10Animal):
    @classmethod
    def describe_class(cls) -> str:
        return super().describe_class() + "; can_fly=depends"


class P10Penguin(P10Bird):
    pass


p10_outputs = [cls.describe_class()
               for cls in (P10Animal, P10Bird, P10Penguin)]
print(*p10_outputs, sep="\n")


species=P10Animal
species=P10Bird; can_fly=depends
species=P10Penguin; can_fly=depends


In [21]:
assert p10_outputs == [
    "species=P10Animal",
    "species=P10Bird; can_fly=depends",
    "species=P10Penguin; can_fly=depends",
]
print("PASS Problem 10: A cooperative `@classmethod` using `super()`")


PASS Problem 10: A cooperative `@classmethod` using `super()`


---

## Part III — Special methods and behavioral contracts

Overriding is a public promise, not merely replacing a function name. These exercises reveal hidden data-model consequences.


### Problem 11 — Return `NotImplemented` in binary operators · Expert

**Concepts:** operator overriding, reflected dispatch, `NotImplemented`

**Your task**

Implement a two-dimensional vector's `__add__`, rejecting unsupported operand types by returning the singleton `NotImplemented`. Test vector addition and the attempted reverse addition of an unrelated type. Explain why raising `TypeError` inside the left operand is often premature.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 11**

For binary operators, returning `NotImplemented` lets Python try the reflected method (`__radd__`) on the right operand before concluding that the operation is unsupported. It is not the same as raising `NotImplementedError`. Use `isinstance` only for the supported vector family and avoid hiding a legitimate reflected implementation.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [22]:
class P11Vector:
    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y

    def __add__(self, other):
        if not isinstance(other, P11Vector):
            return NotImplemented
        return P11Vector(self.x + other.x, self.y + other.y)

    def __repr__(self) -> str:
        return f"P11Vector({self.x}, {self.y})"


class P11Offset:
    def __radd__(self, other):
        if isinstance(other, P11Vector):
            return (other.x + 100, other.y + 100)
        return NotImplemented


p11_total = P11Vector(1, 2) + P11Vector(3, 4)
p11_reflected = P11Vector(1, 2) + P11Offset()
print(repr(p11_total), p11_reflected)


P11Vector(4, 6) (101, 102)


In [23]:
assert (p11_total.x, p11_total.y) == (4, 6)
assert p11_reflected == (101, 102)
assert P11Vector(1, 1).__add__(object()) is NotImplemented
try:
    P11Vector(1, 1) + object()
except TypeError:
    pass
else:
    raise AssertionError("Unsupported addition should ultimately raise TypeError")
print("PASS Problem 11: Return `NotImplemented` in binary operators")


PASS Problem 11: Return `NotImplemented` in binary operators


### Problem 12 — Preserve equality and hashing invariants · Expert

**Concepts:** `__eq__`, `__hash__`, immutability, value semantics

**Your task**

Give a value object an `__eq__` based on its two immutable coordinates and make it usable as a dictionary key. Explain why defining `__eq__` without an appropriate `__hash__` makes normal instances unhashable. Demonstrate unequal unrelated types by returning `NotImplemented`.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 12**

Python normally sets `__hash__ = None` when a class defines its own `__eq__` without defining a hash. Immutable value semantics allow `hash((x, y))`; the hash must agree whenever equality is true. Keep fields read-only to avoid dictionary-key mutation bugs; this example uses `__slots__` and a guarded `__setattr__` to make the contract explicit.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [24]:
class P12Coordinate:
    __slots__ = ("_x", "_y")

    def __init__(self, x: int, y: int) -> None:
        object.__setattr__(self, "_x", x)
        object.__setattr__(self, "_y", y)

    def __setattr__(self, name, value) -> None:
        raise AttributeError("coordinates are immutable")

    @property
    def x(self) -> int:
        return self._x

    @property
    def y(self) -> int:
        return self._y

    def __eq__(self, other):
        if not isinstance(other, P12Coordinate):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self) -> int:
        return hash((self.x, self.y))


p12_a, p12_b = P12Coordinate(2, 3), P12Coordinate(2, 3)
p12_lookup = {p12_a: "found"}
print(p12_lookup[p12_b])


found


In [25]:
assert p12_a == p12_b
assert hash(p12_a) == hash(p12_b)
assert p12_lookup[p12_b] == "found"
assert p12_a.__eq__(object()) is NotImplemented
try:
    p12_a.x = 5
except AttributeError:
    pass
else:
    raise AssertionError("Dictionary keys must not be mutable")
class P12Unhashable:
    def __eq__(self, other):
        return isinstance(other, P12Unhashable)
assert P12Unhashable.__hash__ is None
print("PASS Problem 12: Preserve equality and hashing invariants")


PASS Problem 12: Preserve equality and hashing invariants


### Problem 13 — Abstract overrides and template-method enforcement · Expert

**Concepts:** ABC, abstract override, template method

**Your task**

Create an abstract exporter that owns a public `export` workflow and declares one abstract `serialize` hook. Make a JSON-lines subclass. Test that the abstract base cannot be instantiated, that subclasses work, and that the common workflow is not unnecessarily duplicated.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 13**

`abc.ABC` and `@abstractmethod` make the missing override an instantiation-time error. The nonabstract `export` method centralizes validation and the final newline; only the serialization step varies. This is a practical way to keep subclass behavior within a shared workflow contract.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [26]:
from abc import ABC, abstractmethod
import json


class P13Exporter(ABC):
    @abstractmethod
    def serialize(self, record: dict) -> str:
        """Convert one record into a single line."""

    def export(self, records: list[dict]) -> str:
        if not records:
            return ""
        return "\n".join(self.serialize(record) for record in records) + "\n"


class P13JsonLines(P13Exporter):
    def serialize(self, record: dict) -> str:
        return json.dumps(record, sort_keys=True, separators=(",", ":"))


p13_text = P13JsonLines().export([{"b": 2, "a": 1}, {"x": 0}])
print(p13_text, end="")


{"a":1,"b":2}
{"x":0}


In [27]:
assert p13_text == '{"a":1,"b":2}\n{"x":0}\n'
assert P13JsonLines().export([]) == ""
try:
    P13Exporter()
except TypeError:
    pass
else:
    raise AssertionError("Abstract base must not be instantiated")
print("PASS Problem 13: Abstract overrides and template-method enforcement")


PASS Problem 13: Abstract overrides and template-method enforcement


### Problem 14 — Keep method signatures substitutable · Expert

**Concepts:** Liskov substitution, compatible signatures, optional kwargs

**Your task**

Clients call `render(text, *, uppercase=False)` through a base-class reference. Implement a subclass that adds optional line numbers but **does not** make a new argument required or remove the base keyword. Test all inherited call patterns, including uppercase and direct child behavior.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 14**

A child that replaces an optional parameter with a required one, or removes an accepted keyword, breaks existing clients. Keep the base signature and add only an optional keyword parameter. Delegate the base transformation via `super().render(...)` so improvements to base behavior are preserved.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [28]:
class P14Renderer:
    def render(self, text: str, *, uppercase: bool = False) -> str:
        return text.upper() if uppercase else text


class P14NumberedRenderer(P14Renderer):
    def render(self, text: str, *, uppercase: bool = False,
               line_numbers: bool = False) -> str:
        result = super().render(text, uppercase=uppercase)
        if not line_numbers:
            return result
        return "\n".join(
            f"{index}: {line}"
            for index, line in enumerate(result.split("\n"), start=1)
        )


def p14_client(renderer: P14Renderer) -> tuple[str, str]:
    return renderer.render("hi"), renderer.render("hi", uppercase=True)


p14_child = P14NumberedRenderer()
print(p14_child.render("a\nb", line_numbers=True))


1: a
2: b


In [29]:
assert p14_client(P14Renderer()) == ("hi", "HI")
assert p14_client(p14_child) == ("hi", "HI")
assert p14_child.render("a\nb", line_numbers=True) == "1: a\n2: b"
assert p14_child.render("a", uppercase=True, line_numbers=True) == "1: A"
print("PASS Problem 14: Keep method signatures substitutable")


PASS Problem 14: Keep method signatures substitutable


### Problem 15 — Preserve an exception contract · Expert

**Concepts:** exception contract, validation, side-effect ordering

**Your task**

A payment gateway promises `ValueError` for nonpositive charges. Add an audited gateway that overrides `charge`, logs successful charges once, and keeps invalid transactions off the log. No real payments or credentials are involved.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 15**

Validate and perform the base operation *before* appending to the audit log. `super().charge(amount)` keeps the parent's exception and output contracts intact; storing to the log only on success avoids falsely recording failed charges. Tests cover the happy path and rollback-like failure invariant.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [30]:
from decimal import Decimal


class P15Gateway:
    def charge(self, amount: Decimal) -> str:
        if amount <= 0:
            raise ValueError("amount must be positive")
        return f"charged:{amount:.2f}"


class P15AuditedGateway(P15Gateway):
    def __init__(self) -> None:
        self.audit: list[str] = []

    def charge(self, amount: Decimal) -> str:
        receipt = super().charge(amount)
        self.audit.append(receipt)
        return receipt


p15_gateway = P15AuditedGateway()
print(p15_gateway.charge(Decimal("12.50")))


charged:12.50


In [31]:
assert p15_gateway.audit == ["charged:12.50"]
try:
    p15_gateway.charge(Decimal("0"))
except ValueError as exc:
    assert "positive" in str(exc)
else:
    raise AssertionError("Must retain the base exception")
assert p15_gateway.audit == ["charged:12.50"]
print("PASS Problem 15: Preserve an exception contract")


PASS Problem 15: Preserve an exception contract


### Problem 16 — Override `__format__` with strict format specs · Expert

**Concepts:** `__format__`, f-strings, avoiding recursion

**Your task**

Define a monetary class that supports the empty format spec and `usd`/`plain` modes through `__format__`. An unsupported spec must raise `ValueError`. The `__str__` delegates to an empty-spec format. Verify f-string calls dispatch to the override.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 16**

Formatting has its own special method: `f"{obj:usd}"` calls `type(obj).__format__(obj, "usd")`. Being explicit about supported specs avoids accidental silent formatting. `str` may delegate to `format(self, "")`, but `__format__` must not delegate to `str(self)` in a way that creates recursion.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [32]:
from decimal import Decimal


class P16Money:
    def __init__(self, amount: str) -> None:
        self.amount = Decimal(amount)

    def __format__(self, spec: str) -> str:
        if spec in ("", "plain"):
            return f"{self.amount:.2f}"
        if spec == "usd":
            return f"USD {self.amount:.2f}"
        raise ValueError(f"unsupported money format: {spec!r}")

    def __str__(self) -> str:
        return format(self, "")


p16_money = P16Money("12.5")
print(f"{p16_money}", f"{p16_money:usd}", f"{p16_money:plain}")


12.50 USD 12.50 12.50


In [33]:
assert str(p16_money) == "12.50"
assert f"{p16_money:usd}" == "USD 12.50"
assert format(p16_money, "plain") == "12.50"
try:
    format(p16_money, "unknown")
except ValueError:
    pass
else:
    raise AssertionError("Unsupported format specs must fail")
print("PASS Problem 16: Override `__format__` with strict format specs")


PASS Problem 16: Override `__format__` with strict format specs


---

## Part IV — Multiple inheritance and MRO

Use the actual MRO rather than imagining that `super()` is a synonym for a specific parent name.


### Problem 17 — Trace a cooperative diamond without duplicate calls · Expert

**Concepts:** C3 MRO, cooperative `super()`, diamond inheritance

**Your task**

Construct `Root`, `Left`, `Right`, and `Diamond(Left, Right)` where every `step` logs its class name and calls `super().step()`. Root terminates the chain. Show the exact MRO and exact visitation order; Root must run only once.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 17**

C3 linearization gives `Diamond → Left → Right → Root → object`. When `Left.step` calls `super().step`, the next method in *Diamond's MRO* is `Right.step`, not `Root.step`. Each cooperative method calls `super()` once, and Root ends the chain.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [34]:
class P17Root:
    def step(self, trace: list[str]) -> None:
        trace.append("Root")


class P17Left(P17Root):
    def step(self, trace: list[str]) -> None:
        trace.append("Left")
        super().step(trace)


class P17Right(P17Root):
    def step(self, trace: list[str]) -> None:
        trace.append("Right")
        super().step(trace)


class P17Diamond(P17Left, P17Right):
    def step(self, trace: list[str]) -> None:
        trace.append("Diamond")
        super().step(trace)


p17_trace: list[str] = []
P17Diamond().step(p17_trace)
print([klass.__name__ for klass in P17Diamond.__mro__])
print(p17_trace)


['P17Diamond', 'P17Left', 'P17Right', 'P17Root', 'object']
['Diamond', 'Left', 'Right', 'Root']


In [35]:
assert p17_trace == ["Diamond", "Left", "Right", "Root"]
assert [c.__name__ for c in P17Diamond.__mro__] == [
    "P17Diamond", "P17Left", "P17Right", "P17Root", "object"
]
assert p17_trace.count("Root") == 1
print("PASS Problem 17: Trace a cooperative diamond without duplicate calls")


PASS Problem 17: Trace a cooperative diamond without duplicate calls


### Problem 18 — Cooperative initialization with keyword forwarding · Expert

**Concepts:** cooperative `__init__`, kwargs, fail-fast errors

**Your task**

Implement `NameMixin`, `TimestampMixin`, and `Base` in a diamond-compatible class that accepts only keyword arguments and initializes each field exactly once. Ensure each mixin consumes its own field and forwards everything else through `super().__init__(**kwargs)`. Reject an unexpected keyword.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 18**

Give each class a keyword-only parameter for what it owns, and accept `**kwargs` for the rest. The final base validates no unexpected keywords remain, initializes its own state, then calls `object.__init__()` without kwargs. Every participant calls `super().__init__` exactly once to avoid dropping a sibling initializer.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [36]:
class P18Base:
    def __init__(self, *, identifier: int, **kwargs) -> None:
        if kwargs:
            raise TypeError(f"unexpected arguments: {sorted(kwargs)}")
        self.identifier = identifier
        self.trace = ["Base"]
        super().__init__()


class P18NameMixin(P18Base):
    def __init__(self, *, name: str, **kwargs) -> None:
        super().__init__(**kwargs)
        self.name = name
        self.trace.append("Name")


class P18TimestampMixin(P18Base):
    def __init__(self, *, timestamp: int, **kwargs) -> None:
        super().__init__(**kwargs)
        self.timestamp = timestamp
        self.trace.append("Timestamp")


class P18Record(P18NameMixin, P18TimestampMixin):
    def __init__(self, **kwargs) -> None:
        super().__init__(**kwargs)
        self.trace.append("Record")


p18_record = P18Record(name="Ada", timestamp=42, identifier=7)
print(p18_record.__dict__)


{'identifier': 7, 'trace': ['Base', 'Timestamp', 'Name', 'Record'], 'timestamp': 42, 'name': 'Ada'}


In [37]:
assert (p18_record.name, p18_record.timestamp, p18_record.identifier) == ("Ada", 42, 7)
assert p18_record.trace == ["Base", "Timestamp", "Name", "Record"]
try:
    P18Record(name="Ada", timestamp=42, identifier=7, extra=True)
except TypeError:
    pass
else:
    raise AssertionError("Unknown kwargs must be rejected")
print("PASS Problem 18: Cooperative initialization with keyword forwarding")


PASS Problem 18: Cooperative initialization with keyword forwarding


### Problem 19 — Change mixin order and observe wrapper order · Expert

**Concepts:** mixin composition, wrapper order, MRO

**Your task**

Two mixins wrap `handle(message)` with `A(...)` and `B(...)` and both delegate with `super()`. Implement two endpoint classes with opposite mixin orders. Predict their exact output. Verify that both wrappers execute exactly once and no implementation is copied.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 19**

Each mixin wraps the result from the next class in the MRO. Therefore `Endpoint(A, B, Core)` produces `A(B(core))`, whereas `Endpoint(B, A, Core)` produces `B(A(core))`. Mixins should be small, cooperative, and clear about how ordering changes behavior.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [38]:
class P19Core:
    def handle(self, message: str) -> str:
        return f"core:{message}"


class P19A:
    def handle(self, message: str) -> str:
        return f"A({super().handle(message)})"


class P19B:
    def handle(self, message: str) -> str:
        return f"B({super().handle(message)})"


class P19EndpointAB(P19A, P19B, P19Core):
    pass


class P19EndpointBA(P19B, P19A, P19Core):
    pass


p19_ab = P19EndpointAB().handle("ping")
p19_ba = P19EndpointBA().handle("ping")
print(p19_ab, p19_ba)


A(B(core:ping)) B(A(core:ping))


In [39]:
assert p19_ab == "A(B(core:ping))"
assert p19_ba == "B(A(core:ping))"
assert p19_ab.count("core:") == p19_ba.count("core:") == 1
print("PASS Problem 19: Change mixin order and observe wrapper order")


PASS Problem 19: Change mixin order and observe wrapper order


### Problem 20 — Why a hardcoded parent call breaks a diamond · Expert

**Concepts:** direct-base call versus cooperative `super()`

**Your task**

Construct a sibling-hook hierarchy. Compare a broken left branch that calls `Root.step(self, trace)` directly with a cooperative branch that calls `super().step(trace)`. Show that the hardcoded call bypasses the right sibling even though it is in the MRO.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 20**

A direct `Root.step(self, ...)` chooses one specific method and skips the next MRO participant. `super()` preserves cooperative traversal. Hardcoded base calls can be appropriate for deliberately non-cooperative operations, but not when the hierarchy promises cooperative multiple inheritance.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [40]:
class P20Root:
    def step(self, trace: list[str]) -> None:
        trace.append("root")


class P20Right(P20Root):
    def step(self, trace: list[str]) -> None:
        trace.append("right")
        super().step(trace)


class P20BrokenLeft(P20Root):
    def step(self, trace: list[str]) -> None:
        trace.append("broken-left")
        P20Root.step(self, trace)  # deliberately wrong for a cooperative diamond


class P20GoodLeft(P20Root):
    def step(self, trace: list[str]) -> None:
        trace.append("good-left")
        super().step(trace)


class P20Broken(P20BrokenLeft, P20Right):
    pass


class P20Fixed(P20GoodLeft, P20Right):
    pass


p20_broken, p20_fixed = [], []
P20Broken().step(p20_broken)
P20Fixed().step(p20_fixed)
print("broken:", p20_broken)
print("fixed:", p20_fixed)


broken: ['broken-left', 'root']
fixed: ['good-left', 'right', 'root']


In [41]:
assert p20_broken == ["broken-left", "root"]
assert p20_fixed == ["good-left", "right", "root"]
assert P20Right in P20Broken.__mro__
assert p20_fixed.count("root") == 1
print("PASS Problem 20: Why a hardcoded parent call breaks a diamond")


PASS Problem 20: Why a hardcoded parent call breaks a diamond


### Problem 21 — Resolve an inconsistent hierarchy intentionally · Expert

**Concepts:** C3 consistency, incompatible parent orders, design correction

**Your task**

Explain why Python rejects a hypothetical `Combined(A, B)` if `A` requires `X` before `Y` and `B` requires `Y` before `X`. Write minimal code that catches the class-definition-time `TypeError`, then design a consistent shared ordering that is constructible.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 21**

C3 MRO must preserve each parent's ordering. `A(X, Y)` imposes `X < Y`; `B(Y, X)` imposes `Y < X`; no linear order can satisfy both. Fix the *class design* rather than overriding `mro()` to conceal the conflict; make both branches agree on base order.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [42]:
class P21X:
    pass


class P21Y:
    pass


class P21A(P21X, P21Y):
    pass


class P21B(P21Y, P21X):
    pass


p21_error = ""
try:
    class P21Impossible(P21A, P21B):
        pass
except TypeError as exc:
    p21_error = str(exc)


class P21ConsistentB(P21X, P21Y):
    pass


class P21Possible(P21A, P21ConsistentB):
    pass


print("Conflict:", p21_error)
print("Consistent MRO:", [c.__name__ for c in P21Possible.__mro__])


Conflict: Cannot create a consistent method resolution order (MRO) for bases P21X, P21Y
Consistent MRO: ['P21Possible', 'P21A', 'P21ConsistentB', 'P21X', 'P21Y', 'object']


In [43]:
assert p21_error and "resolution order" in p21_error
assert P21Possible.__mro__.index(P21X) < P21Possible.__mro__.index(P21Y)
assert P21A in P21Possible.__mro__ and P21ConsistentB in P21Possible.__mro__
print("PASS Problem 21: Resolve an inconsistent hierarchy intentionally")


PASS Problem 21: Resolve an inconsistent hierarchy intentionally


---

## Part V — Python data-model traps and modern conveniences

These exercises address overriding surprises that are easy to miss when reading only ordinary method calls.


### Problem 22 — Instance monkey-patching does not override `str` · Expert

**Concepts:** special method lookup, type versus instance

**Your task**

Assign a callable named `__str__` directly onto *one instance*, then compare `obj.__str__()` and `str(obj)`. Afterward make a genuine subclass that overrides `__str__`. Show which strategy Python's special-method protocol respects.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 22**

Implicit special-method lookup generally begins at the type, not on the instance dictionary. An instance `obj.__str__` can shadow an *explicit* attribute lookup, but `str(obj)` still invokes the type's special method. Subclassing and overriding on the class is the correct supported customization.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [44]:
class P22Base:
    def __str__(self) -> str:
        return "class implementation"


p22_object = P22Base()
p22_object.__str__ = lambda: "instance patch"  # deliberate trap


class P22Child(P22Base):
    def __str__(self) -> str:
        return "child override"


print(p22_object.__str__(), "|", str(p22_object))
print(str(P22Child()))


instance patch | class implementation
child override


In [45]:
assert p22_object.__str__() == "instance patch"
assert str(p22_object) == "class implementation"
assert str(P22Child()) == "child override"
assert "__str__" in p22_object.__dict__
print("PASS Problem 22: Instance monkey-patching does not override `str`")


PASS Problem 22: Instance monkey-patching does not override `str`


### Problem 23 — Overriding `__getattribute__` safely · Expert

**Concepts:** `__getattribute__`, safe delegation, recursion risk

**Your task**

Instrument every ordinary attribute access without accidentally recursing forever. A child should log public field names, then delegate to the parent using `super().__getattribute__(name)`. Prove that reading the internal log does not recursively log itself. Explain why directly using `self.audit` inside this method is dangerous.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 23**

`__getattribute__` intercepts nearly every normal attribute access. Accessing `self.audit` from inside it would recursively invoke the same override. Use `object.__getattribute__(self, "_audit")` for internal bookkeeping, and `super().__getattribute__(name)` for the original lookup semantics. This is specialized machinery; ordinary properties are usually simpler.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [46]:
class P23Base:
    def __init__(self) -> None:
        object.__setattr__(self, "_audit", [])
        self.value = 42


class P23Instrumented(P23Base):
    def __getattribute__(self, name: str):
        if not name.startswith("_"):
            audit = object.__getattribute__(self, "_audit")
            audit.append(name)
        return super().__getattribute__(name)


p23_item = P23Instrumented()
p23_value = p23_item.value
p23_audit = object.__getattribute__(p23_item, "_audit")
print(p23_value, p23_audit)


42 ['value']


In [47]:
assert p23_value == 42
assert p23_audit == ["value"]
assert p23_item.value == 42
assert p23_audit == ["value", "value"]
assert object.__getattribute__(p23_item, "_audit") is p23_audit
print("PASS Problem 23: Overriding `__getattribute__` safely")


PASS Problem 23: Overriding `__getattribute__` safely


### Problem 24 — Dataclass-generated methods versus inherited methods · Expert

**Concepts:** dataclass generation, inherited explicit `__str__`

**Your task**

Create a parent with a custom `__repr__` and `__str__`. Derive two dataclasses: one using default `repr=True`, and another using `repr=False`. Predict their `repr` and `str` results. Explain what gets generated and which inherited methods remain intact.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 24**

With default `repr=True`, the dataclass decorator generates a `__repr__` on the decorated subclass; with `repr=False`, it leaves the parent `__repr__` available. Neither case generates `__str__`, so the inherited explicit `__str__` still wins in both classes. This is a powerful counterexample to an overbroad 'child repr always controls str' rule.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [48]:
from dataclasses import dataclass


class P24Base:
    def __repr__(self) -> str:
        return "parent repr"

    def __str__(self) -> str:
        return "parent str"


@dataclass
class P24Generated(P24Base):
    value: int


@dataclass(repr=False)
class P24Inherited(P24Base):
    value: int


p24_generated, p24_inherited = P24Generated(3), P24Inherited(3)
print(repr(p24_generated), str(p24_generated))
print(repr(p24_inherited), str(p24_inherited))


P24Generated(value=3) parent str
parent repr parent str


In [49]:
assert repr(p24_generated) == "P24Generated(value=3)"
assert str(p24_generated) == "parent str"
assert repr(p24_inherited) == "parent repr"
assert str(p24_inherited) == "parent str"
assert "__repr__" in P24Generated.__dict__
assert "__repr__" not in P24Inherited.__dict__
print("PASS Problem 24: Dataclass-generated methods versus inherited methods")


PASS Problem 24: Dataclass-generated methods versus inherited methods


### Problem 25 — Redact sensitive data in debugging representations · Expert

**Concepts:** safe `__repr__`, security-minded debugging, container behavior

**Your task**

Create a credential record with a secret token. Its `__repr__` and `__str__` must not reveal the token, including when the record is inside a list. Provide a safe informative identifier and show a controlled token-validation operation without printing sensitive data.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 25**

Debugging representations commonly end up in logs, tracebacks, and container displays. Both methods must avoid exposing secrets. Write one safe `__repr__` and let `__str__` delegate to it; centralizing redaction prevents divergence. The token is stored only to support a trivial equality check in this educational example; real credential handling requires stronger controls.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [50]:
class P25Credential:
    def __init__(self, user: str, token: str) -> None:
        self.user = user
        self._token = token

    def __repr__(self) -> str:
        return f"P25Credential(user={self.user!r}, token=<redacted>)"

    def __str__(self) -> str:
        return repr(self)

    def matches(self, provided_token: str) -> bool:
        return self._token == provided_token


p25_credential = P25Credential("user-7", "DO-NOT-PRINT-THIS")
p25_displays = (str(p25_credential), repr(p25_credential),
                repr([p25_credential]))
print(*p25_displays, sep="\n")


P25Credential(user='user-7', token=<redacted>)
P25Credential(user='user-7', token=<redacted>)
[P25Credential(user='user-7', token=<redacted>)]


In [51]:
assert all("DO-NOT-PRINT-THIS" not in text for text in p25_displays)
assert all("<redacted>" in text for text in p25_displays)
assert p25_credential.matches("DO-NOT-PRINT-THIS")
assert not p25_credential.matches("incorrect")
print("PASS Problem 25: Redact sensitive data in debugging representations")


PASS Problem 25: Redact sensitive data in debugging representations


### Problem 26 — Respect subtype fields when implementing `__repr__` · Expert

**Concepts:** subclass representation, virtual hook, base initialization

**Your task**

A base `Event` has `name`; a child `TimedEvent` adds `seconds`. Design representations that identify the correct runtime type and do not accidentally omit child fields. Also implement a common `display` method that calls a dynamically dispatched `details` hook.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 26**

A blindly inherited base `repr` can underreport subclass state. Override it in a child when its constructor/state differs, and ensure it includes the subclass's own fields. A common presentation method need not be duplicated: it can call a hook through `self`, exactly like the source lesson's inherited `extended_info` example.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [52]:
class P26Event:
    def __init__(self, name: str) -> None:
        self.name = name

    def __repr__(self) -> str:
        return f"P26Event(name={self.name!r})"

    def details(self) -> str:
        return self.name

    def display(self) -> str:
        return f"Event: {self.details()}"


class P26TimedEvent(P26Event):
    def __init__(self, name: str, seconds: int) -> None:
        super().__init__(name)
        if seconds < 0:
            raise ValueError("seconds must be nonnegative")
        self.seconds = seconds

    def __repr__(self) -> str:
        return f"P26TimedEvent(name={self.name!r}, seconds={self.seconds!r})"

    def details(self) -> str:
        return f"{self.name} at {self.seconds}s"


p26_event = P26TimedEvent("alarm", 90)
print(repr(p26_event))
print(p26_event.display())


P26TimedEvent(name='alarm', seconds=90)
Event: alarm at 90s


In [53]:
assert repr(p26_event) == "P26TimedEvent(name='alarm', seconds=90)"
assert p26_event.display() == "Event: alarm at 90s"
assert P26Event("start").display() == "Event: start"
try:
    P26TimedEvent("bad", -1)
except ValueError:
    pass
else:
    raise AssertionError("Invalid seconds must be rejected")
print("PASS Problem 26: Respect subtype fields when implementing `__repr__`")


PASS Problem 26: Respect subtype fields when implementing `__repr__`


---

## Part VI — Integrative capstones

Combine the principles in larger but still testable applications. These solutions deliberately include multiple classes, edge cases, and end-to-end checks.


### Problem 27 — Capstone: pluggable report pipeline · Expert capstone

**Concepts:** template method, hooks, safe CSV, audit after success

**Your task**

Design a `P27Report` template: its public `generate(data)` validates input once, asks overridable `title()` and `render_body()` hooks for content, and returns a complete report. Provide `TextReport` and `CsvReport` subclasses. Add an `AuditedTextReport` that uses `super().generate(...)` and records only successful runs. Test valid and empty data, invalid input, and that shared logic is not duplicated.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 27**

Use one nonduplicated template method for validation and structure. Subclasses specialize hooks; the audit subclass extends the workflow through `super()` rather than copying it. Append an audit entry *after* successful generation. To keep CSV safe and correct for commas and quotes, use the standard `csv` module rather than hand-joining fields.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [54]:
import csv
import io


class P27Report:
    def title(self) -> str:
        raise NotImplementedError

    def render_body(self, data: list[tuple[str, int]]) -> str:
        raise NotImplementedError

    def generate(self, data: list[tuple[str, int]]) -> str:
        if not isinstance(data, list):
            raise TypeError("data must be a list")
        if any(not isinstance(item, tuple) or len(item) != 2
               or not isinstance(item[0], str) or not isinstance(item[1], int)
               for item in data):
            raise ValueError("records must be (str, int) tuples")
        return f"{self.title()}\n{self.render_body(data)}"


class P27TextReport(P27Report):
    def title(self) -> str:
        return "TEXT REPORT"

    def render_body(self, data: list[tuple[str, int]]) -> str:
        return "\n".join(f"{label}: {number}" for label, number in data)


class P27CsvReport(P27Report):
    def title(self) -> str:
        return "CSV REPORT"

    def render_body(self, data: list[tuple[str, int]]) -> str:
        stream = io.StringIO(newline="")
        writer = csv.writer(stream, lineterminator="\n")
        writer.writerow(("label", "value"))
        writer.writerows(data)
        return stream.getvalue().rstrip("\n")


class P27AuditedTextReport(P27TextReport):
    def __init__(self) -> None:
        self.completed_runs = 0

    def generate(self, data: list[tuple[str, int]]) -> str:
        result = super().generate(data)
        self.completed_runs += 1
        return result


p27_report = P27AuditedTextReport()
p27_text = p27_report.generate([("apples", 3)])
p27_csv = P27CsvReport().generate([("a,b", 2), ('"quoted"', 4)])
print(p27_text)
print(p27_csv)


TEXT REPORT
apples: 3
CSV REPORT
label,value
"a,b",2
"""quoted""",4


In [55]:
assert p27_text == "TEXT REPORT\napples: 3"
assert p27_csv == 'CSV REPORT\nlabel,value\n"a,b",2\n"""quoted""",4'
assert P27TextReport().generate([]) == "TEXT REPORT\n"
assert p27_report.completed_runs == 1
for bad in (None, [("bad", "3")], ["not a pair"]):
    try:
        p27_report.generate(bad)
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError("Malformed data must fail")
assert p27_report.completed_runs == 1
print("PASS Problem 27: Capstone: pluggable report pipeline")


PASS Problem 27: Capstone: pluggable report pipeline


### Problem 28 — Capstone: auditable cooperative message pipeline · Expert capstone

**Concepts:** cooperative mixins, validation, normalization, audit-on-success

**Your task**

Create a reusable base `send` operation that validates a message and returns a delivery string. Add two cooperative mixins: one strips surrounding whitespace, the other records each valid delivery exactly once. Compose an endpoint with a deliberate MRO, and provide a second ordering to show why order matters. Test clean input, whitespace-only rejection, audit behavior, and the MRO.

**Hint:** Follow method lookup using the runtime type of `self`; distinguish the method *containing* a call from the object on which it is dispatched.

*Try implementing or predicting the behavior before expanding the next solution.*


**Worked solution — Problem 28**

The normalization mixin first transforms the message then delegates. The auditing mixin delegates and records the successfully returned value plus its own incoming argument. With order `(Normalize, Audit, Base)`, that argument has been normalized. With `(Audit, Normalize, Base)`, the audit observes the original string, although the final delivery receipt is normalized in both cases. Recording is still performed only after success. Each mixin's initializer also delegates cooperatively.

The following implementation and its separate assertion cell are executable and require only Python’s standard library.


In [56]:
class P28Base:
    def __init__(self, **kwargs) -> None:
        if kwargs:
            raise TypeError(f"Unexpected keywords: {sorted(kwargs)}")
        self.deliveries: list[str] = []
        super().__init__()

    def send(self, message: str) -> str:
        if not isinstance(message, str) or not message.strip():
            raise ValueError("message must contain non-whitespace text")
        receipt = f"delivered:{message}"
        self.deliveries.append(receipt)
        return receipt


class P28NormalizeMixin:
    def __init__(self, **kwargs) -> None:
        super().__init__(**kwargs)

    def send(self, message: str) -> str:
        if not isinstance(message, str):
            raise ValueError("message must be text")
        return super().send(message.strip())


class P28AuditMixin:
    def __init__(self, **kwargs) -> None:
        super().__init__(**kwargs)
        self.audit: list[str] = []

    def send(self, message: str) -> str:
        receipt = super().send(message)
        self.audit.append(receipt)
        return receipt


class P28Endpoint(P28NormalizeMixin, P28AuditMixin, P28Base):
    pass


class P28Alternative(P28AuditMixin, P28NormalizeMixin, P28Base):
    pass


p28_endpoint = P28Endpoint()
p28_alternative = P28Alternative()
p28_first = p28_endpoint.send("   hello  ")
p28_second = p28_alternative.send("   hello  ")
print(p28_first, p28_second)
print([cls.__name__ for cls in P28Endpoint.__mro__])


delivered:hello delivered:hello
['P28Endpoint', 'P28NormalizeMixin', 'P28AuditMixin', 'P28Base', 'object']


In [57]:
assert p28_first == p28_second == "delivered:hello"
for endpoint in (p28_endpoint, p28_alternative):
    assert endpoint.audit == ["delivered:hello"]
    assert endpoint.deliveries == ["delivered:hello"]
    for bad in ("  ", None):
        try:
            endpoint.send(bad)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid messages must be rejected")
    assert len(endpoint.audit) == len(endpoint.deliveries) == 1
assert [klass.__name__ for klass in P28Endpoint.__mro__] == [
    "P28Endpoint", "P28NormalizeMixin", "P28AuditMixin", "P28Base", "object"
]
print("PASS Problem 28: Capstone: auditable cooperative message pipeline")


PASS Problem 28: Capstone: auditable cooperative message pipeline


---

## Final review — self-check with answers

| Question | Concise correct answer |
|---|---|
| A child defines only `__repr__`; does `str(child)` always use it? | **No.** It does when the effective `__str__` is `object.__str__` or an explicit method that delegates to `repr`, but not when it inherits a nondelegating `__str__`. |
| Why can a parent method call a child override? | `self` is the runtime object, so `self.hook()` performs lookup from its actual class. |
| Does `super()` always call the lexical parent? | **No.** Lookup continues after the method's defining class within the runtime class's MRO. |
| Why call `super().__init__()`? | To preserve parent initialization and, in a cooperative hierarchy, let every MRO participant initialize its state. |
| What should an unsupported `__add__` return? | `NotImplemented`, so Python can try reflected dispatch. |
| Does attaching `obj.__str__ = ...` change `str(obj)`? | **No.** Implicit special-method lookup generally uses the object's type. |
| Why must `__eq__` and `__hash__` agree? | Equal objects must have equal hashes; mutable hashed fields would break dictionary/set lookups. |
| Where should an audit log be updated? | After the delegated operation succeeds, unless the contract explicitly requires recording failed attempts. |

**Study checklist:** can you (1) hand-trace an MRO, (2) distinguish an inherited explicit method from an `object` fallback, (3) predict where dynamic calls on `self` land, (4) design a cooperative diamond, (5) write a nonrecursive override, and (6) validate success and failure paths with assertions?

**Practical next exercise (answer provided):** implement `class Custom(P19A, P19Core): pass`; `Custom().handle("x")` evaluates to `"A(core:x)"` because `super()` proceeds from `P19A` to `P19Core` in that class's MRO. Try checking this interactively with the definitions from Problem 19.

**Version note:** this notebook targets Python 3.10+ for built-in generic type hints; the underlying dispatch concepts apply to earlier Python 3 versions, including the older lecture source, with appropriate syntax adjustments.
